# 모의고사 10. 마케팅 전환(구매) 예측 (이진분류, 불균형)

> 실제 시험처럼 이론 설명 없이 문제만 순서대로 제시합니다. **90분 타이머를 맞추고** 풀어보세요. 오픈북 허용 문서(numpy/pandas/matplotlib/seaborn/scikit-learn/tensorflow/XGBoost 공식문서)만 참고할 수 있다고 가정합니다.

### 데이터 소개
`data/marketing_conversion.csv` - 광고 노출 고객의 행동 데이터(방문횟수, 클릭수, 이전구매금액 등)로 구매여부(약 28%:72% 불균형)를 예측합니다.

> ⏱️ **[실전 타임어택 가이드]** 데이터 탐색 10분 ➔ 전처리 20분 ➔ 머신러닝 30분 ➔ 딥러닝 30분
> 막히는 부분은 주석으로 남기고 다음 문제로 넘어가는 것이 합격 전략입니다.


## 목차
1교시 데이터 로딩 & EDA · 2교시 데이터 시각화 · 3교시 데이터 전처리 · 4교시 머신러닝 모델링 · 5교시 딥러닝 모델링 · 채점 가이드

In [ ]:
import sys
sys.path.insert(0, '../00_시작하기')
import aice_grader as grader

# 실전처럼 시간 제한(90분)을 두고 풀어보세요. 아래 셀을 실행하면 타이머가 시작됩니다.
timer = grader.ExamTimer(minutes=90)
timer.start()

## 1교시: 데이터 로딩 & EDA

**문제 1.** `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn`을 각각 `pd`, `np`, `plt`, `sns`로 불러오고, `../data/marketing_conversion.csv`을 `df`로 불러와 shape을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요

<details>
<summary>✅ 정답 보기</summary>

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/marketing_conversion.csv')
df.head()
print(df.shape)
```

</details>

**문제 2.** `df`의 결측치와 `구매여부` 분포를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(df.isnull().sum())
print(df['구매여부'].value_counts(normalize=True))  # 범주형 데이터의 항목별 개수를 세어 데이터 분포 확인
```

</details>

## 2교시: 데이터 시각화

**문제 3.** `기기종류`별 `구매여부`를 countplot(hue)으로 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.countplot(x='기기종류', hue='구매여부', data=df)  # 범주형 변수의 항목별 개수를 막대 그래프로 시각화
plt.show()
```

</details>

**문제 4.** `이전구매금액`을 `구매여부`별로 kdeplot으로 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.kdeplot(data=df, x='이전구매금액', hue='구매여부')  # 데이터의 분포를 부드러운 곡선(밀도 함수)으로 시각화
plt.show()
```

</details>

## 3교시: 데이터 전처리

**문제 5.** `나이`의 결측치를 평균으로 채우세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df['나이'] = df['나이'].fillna(df['나이'].mean())  # 결측치를 지정한 값(평균, 중앙값 등)으로 안전하게 대체
```

</details>

**문제 6.** `기기종류`, `유입경로`를 `get_dummies(drop_first=True)`로 인코딩하고, `train_test_split(test_size=0.2, stratify=y, random_state=42)`로 분할한 뒤 스케일링하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df_enc = pd.get_dummies(df, columns=['기기종류', '유입경로'], drop_first=True)  # 문자로 된 범주형 데이터를 기계가 이해할 수 있는 0과 1 형태(원-핫 인코딩)로 변환
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
X = df_enc.drop(columns=['구매여부'])
y = df_enc['구매여부']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # 데이터를 모델 학습용(train)과 검증용(test)으로 분리
scaler = StandardScaler()  # 데이터를 평균 0, 표준편차 1이 되도록 스케일링 (거리 기반, 딥러닝 알고리즘에 필수)
X_train_s = scaler.fit_transform(X_train)  # 훈련 데이터로 스케일링 기준을 학습(fit)하고 변환(transform)까지 수행
X_test_s = scaler.transform(X_test)  # 훈련 데이터에서 학습된 기준을 그대로 적용하여 평가 데이터를 변환 (데이터 누수 방지)
```

</details>

## 4교시: 머신러닝 모델링

**문제 7.** `LogisticRegression(class_weight='balanced', max_iter=1000)`을 학습시켜 recall, precision, f1을 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import recall_score, precision_score, f1_score
logi = LogisticRegression(class_weight='balanced', max_iter=1000)  # 주로 이진 분류에 쓰이는 로지스틱 회귀 모델 생성
logi.fit(X_train_s, y_train)
pred = logi.predict(X_test_s)
print(recall_score(y_test, pred))
print(precision_score(y_test, pred))
print(f1_score(y_test, pred))  # 정밀도와 재현율의 조화 평균, 데이터가 불균형할 때 중요한 평가지표
```

</details>

**문제 8.** `RandomForestClassifier(class_weight='balanced', random_state=42)`와 성능을 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(class_weight='balanced', random_state=42)  # 여러 개의 의사결정 나무를 묶어 예측하는 랜덤 포레스트 분류 모델 생성
rf.fit(X_train_s, y_train)
print(recall_score(y_test, rf.predict(X_test_s)))
```

</details>

## 5교시: 딥러닝 모델링

**문제 9.** 이진분류 DL 모델(출력층 1노드, sigmoid)을 만들어 학습시키고 confusion matrix를 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import confusion_matrix
model = Sequential()  # 딥러닝 층(Layer)을 순서대로 쌓기 위한 기본 틀(모델) 생성
model.add(Dense(16, activation='relu', input_shape=(X_train_s.shape[1],)))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dense(8, activation='relu'))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.add(Dense(1, activation='sigmoid'))  # 이전 층의 모든 노드와 연결되는 완전 연결층(Dense Layer) 추가
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])  # 모델 학습 과정(최적화 기법, 손실 함수, 평가지표) 설정
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)  # 성능이 개선되지 않으면 무의미한 학습을 일찍 멈춤
model.fit(X_train_s, y_train, epochs=100, batch_size=32, validation_data=(X_test_s, y_test), callbacks=[es], verbose=0)
pred_dl = (model.predict(X_test_s).flatten() > 0.5).astype(int)
cm = confusion_matrix(y_test, pred_dl)  # 실제 정답과 예측 결과를 교차 표(오차 행렬)로 확인
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')  # 상관계수 등을 색상으로 표현하여 한눈에 파악
plt.show()
```

</details>

In [ ]:
# 문제를 다 풀었다면 아래 셀을 실행해 소요 시간을 확인하세요.
timer.finish()

## 채점 가이드 (100점 만점 배점표)

이 모의고사는 총 9문제, 100점 만점입니다. 각 문제를 정답과 비교해 맞았으면 배점만큼, 틀렸으면 0점으로 스스로 채점해보세요.

| 문항 | 배점 |
|---|---|
| 문제 1 | 11점 |
| 문제 2 | 11점 |
| 문제 3 | 11점 |
| 문제 4 | 11점 |
| 문제 5 | 11점 |
| 문제 6 | 11점 |
| 문제 7 | 11점 |
| 문제 8 | 11점 |
| 문제 9 | 12점 |
| **합계** | **100점** |

> 💡 **합격 기준: 80점 이상** (실제 AICE Associate와 동일한 기준입니다)

### 자동으로 기록하며 채점하고 싶다면

`00_시작하기/aice_grader.py`의 `MockExamGrader`를 사용하면 점수를 자동으로 합산해줍니다.

```python
import aice_grader as grader

g = grader.MockExamGrader("모의고사10_마케팅전환예측")
g.check(1, points=11, is_correct=True)   # 문제 1을 맞았으면 True, 틀렸으면 False
g.check(2, points=11, is_correct=True)
# ... 문제 수만큼 반복 ...
g.report()   # 최종 점수와 합격 여부를 출력
```

### 개념 체크리스트

다 풀었다면 아래 체크리스트로 한 번 더 점검해보세요.

- [ ] 라이브러리를 정확한 alias(pd, np, plt, sns 등)로 불러왔는가
- [ ] 결측치를 처리했는가
- [ ] 불균형 데이터에 class_weight를 적용했는가
- [ ] recall/precision/f1을 모두 확인했는가
